# Leveraging Clustering for Large-Scale Time Series Forecasting

**KDD Course Project -- University of Vienna**

---

This notebook performs clustering analysis on household electricity consumption data (17,547 households, daily readings for 2023). The goal is to identify natural consumption segments that can later inform cluster-specific forecasting models.

**Pipeline overview:**
1. Data Loading & Exploration
2. Data Preprocessing (outlier analysis, normalization strategies)
3. Feature Extraction (16 time series features per household)
4. Clustering (K-Means++, Hierarchical, DBSCAN)
5. Results & Analysis (cluster profiles, t-SNE, heatmaps)
6. Summary & Findings

## Setup

Mount Google Drive and configure the data path. Upload `sample_23.csv` to the specified directory on your Google Drive before running.

In [ ]:
# Mount Google Drive (Colab only)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install any missing packages
!pip install -q scikit-learn seaborn pandas numpy matplotlib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

# ----- DATA PATH CONFIGURATION -----
# Option 1: Google Drive (default for Colab)
DATA_DIR = '/content/drive/MyDrive/KDD_Project/data/'

# Option 2: Local file upload (uncomment if not using Drive)
# from google.colab import files
# uploaded = files.upload()  # upload sample_23.csv
# DATA_DIR = '/content/'

# Option 3: Local machine
# DATA_DIR = './data/'

RANDOM_STATE = 42
K_RANGE = range(2, 16)  # k=2 to k=15

print(f"Data directory: {DATA_DIR}")
print("Setup complete.")

---

## 1. Data Loading & Exploration

The dataset contains daily electricity consumption (kWh) for 17,547 households over the full year 2023. Each row is a household, each column is a date.

In [ ]:
# Load the dataset
df = pd.read_csv(DATA_DIR + 'sample_23.csv')
ids = df['ID']
ts = df.drop(columns=['ID'])  # time series matrix: (households, days)
dates = pd.to_datetime(ts.columns)

print(f"Shape: {ts.shape[0]} households x {ts.shape[1]} days")
print(f"Date range: {dates.min().date()} to {dates.max().date()}")
print(f"Missing values: {ts.isna().sum().sum()}")
print()
df.head()

### Basic Statistics

Compute per-household summary statistics to understand the consumption distribution.

In [ ]:
# Per-household statistics
household_mean = ts.mean(axis=1)
household_std = ts.std(axis=1)
household_sum = ts.sum(axis=1)
household_max = ts.max(axis=1)
household_min = ts.min(axis=1)

stats_summary = pd.DataFrame({
    'Metric': [
        'Number of households',
        'Number of days',
        'Mean consumption (kWh/day)',
        'Median consumption (kWh/day)',
        'Std deviation (kWh/day)',
        'Min household mean',
        'Max household mean',
        'Zero-consumption households',
        'Near-zero (<0.1 kWh) households',
    ],
    'Value': [
        ts.shape[0],
        ts.shape[1],
        round(household_mean.mean(), 2),
        round(household_mean.median(), 2),
        round(household_mean.std(), 2),
        round(household_mean.min(), 4),
        round(household_mean.max(), 2),
        int((household_sum == 0).sum()),
        int((household_mean < 0.1).sum()),
    ]
})
stats_summary

In [ ]:
# Percentile distribution of household mean consumption
percentiles = [1, 5, 25, 50, 75, 95, 99]
pct_df = pd.DataFrame({
    'Percentile': [f'{p}th' for p in percentiles],
    'kWh/day': [round(np.percentile(household_mean, p), 3) for p in percentiles]
})
pct_df

### Distribution of Mean Consumption

The distribution is heavily right-skewed (mean >> median), indicating that most households have moderate consumption while a small fraction consumes much more.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
axes[0].hist(household_mean, bins=100, edgecolor='black', alpha=0.7)
axes[0].axvline(household_mean.median(), color='red', linestyle='--',
                label=f'Median = {household_mean.median():.2f}')
axes[0].axvline(household_mean.mean(), color='orange', linestyle='--',
                label=f'Mean = {household_mean.mean():.2f}')
axes[0].set_xlabel('Mean daily consumption (kWh)')
axes[0].set_ylabel('Number of households')
axes[0].set_title('Distribution of Mean Daily Consumption')
axes[0].legend()

# Log scale for better tail visibility
axes[1].hist(household_mean, bins=100, edgecolor='black', alpha=0.7)
axes[1].axvline(np.percentile(household_mean, 99), color='red', linestyle='--',
                label=f'99th pctl = {np.percentile(household_mean, 99):.1f}')
axes[1].set_xlabel('Mean daily consumption (kWh)')
axes[1].set_ylabel('Number of households (log scale)')
axes[1].set_title('Distribution (log y-axis)')
axes[1].set_yscale('log')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot of household mean consumption
fig, ax = plt.subplots(figsize=(10, 3))
ax.boxplot(household_mean, vert=False, widths=0.6)
ax.set_xlabel('Mean daily consumption (kWh)')
ax.set_title('Boxplot of Household Mean Daily Consumption')
plt.tight_layout()
plt.show()

### Outlier Identification

Using the IQR method on household mean consumption to identify outliers.

In [ ]:
Q1 = household_mean.quantile(0.25)
Q3 = household_mean.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

n_outliers = int(((household_mean < lower_bound) | (household_mean > upper_bound)).sum())

print(f"Q1 = {Q1:.2f}, Q3 = {Q3:.2f}, IQR = {IQR:.2f}")
print(f"Lower bound = {lower_bound:.2f}, Upper bound = {upper_bound:.2f}")
print(f"Outliers: {n_outliers} households ({n_outliers / len(household_mean) * 100:.1f}%)")
print(f"\n1st percentile: {household_mean.quantile(0.01):.3f} kWh/day")
print(f"99th percentile: {household_mean.quantile(0.99):.3f} kWh/day")

### Temporal Patterns

Aggregate daily consumption profile across all households and monthly averages.

In [ ]:
# Aggregate daily consumption profile
daily_avg = ts.mean(axis=0)
daily_median = ts.median(axis=0)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(dates, daily_avg, label='Mean', alpha=0.8, linewidth=0.8)
ax.plot(dates, daily_median, label='Median', alpha=0.8, linewidth=0.8)
ax.fill_between(dates,
                ts.quantile(0.25, axis=0),
                ts.quantile(0.75, axis=0),
                alpha=0.2, label='IQR (25th-75th)')
ax.set_xlabel('Date')
ax.set_ylabel('Daily consumption (kWh)')
ax.set_title('Aggregate Daily Consumption Profile (2023)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Monthly average consumption
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

monthly_means = []
for m in range(1, 13):
    cols = [c for c, d in zip(ts.columns, dates) if d.month == m]
    monthly_means.append(ts[cols].mean(axis=1).mean())

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(month_names, monthly_means, color='steelblue', edgecolor='black')
ax.set_ylabel('Mean daily consumption (kWh)')
ax.set_title('Average Daily Consumption by Month (2023)')
plt.tight_layout()
plt.show()

In [ ]:
# Sample time series from different quantiles
fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
quantiles = [0.05, 0.25, 0.50, 0.75, 0.95, 0.99]

for i, q in enumerate(quantiles):
    ax = axes[i // 2, i % 2]
    target_val = household_mean.quantile(q)
    idx = (household_mean - target_val).abs().idxmin()
    ax.plot(dates, ts.loc[idx], linewidth=0.7)
    ax.set_title(f'Household at {q*100:.0f}th percentile '
                 f'(mean={household_mean.loc[idx]:.2f} kWh)')
    ax.set_ylabel('kWh')

axes[-1, 0].set_xlabel('Date')
axes[-1, 1].set_xlabel('Date')
plt.suptitle('Sample Household Consumption Profiles', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Monthly correlation heatmap
monthly_data = pd.DataFrame()
for m in range(1, 13):
    cols = [c for c, d in zip(ts.columns, dates) if d.month == m]
    monthly_data[month_names[m - 1]] = ts[cols].mean(axis=1)

fig, ax = plt.subplots(figsize=(10, 8))
corr = monthly_data.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax,
            vmin=0.5, vmax=1.0)
ax.set_title('Correlation Between Monthly Consumption')
plt.tight_layout()
plt.show()

---

## 2. Data Preprocessing

We apply several normalization strategies to the raw time series. The goal is to evaluate which preprocessing best serves clustering:

| Method | Effect |
|---|---|
| **Raw** | Preserves absolute consumption levels |
| **Z-score** (per household) | Centers to mean=0, std=1; removes scale, preserves shape |
| **Min-Max** (per household) | Scales to [0, 1]; removes scale, preserves shape |
| **Log** (log1p) | Reduces right-skew, compresses high values |
| **Log + Z-score** | Combines skew reduction with normalization |

In [ ]:
# Z-score normalization (per household)
ts_zscore = ts.sub(household_mean, axis=0).div(household_std.replace(0, 1), axis=0)

# Min-max normalization (per household)
ts_range = (household_max - household_min).replace(0, 1)
ts_minmax = ts.sub(household_min, axis=0).div(ts_range, axis=0)

# Log transformation (log1p to handle zeros)
ts_log = np.log1p(ts)

# Log + Z-score
log_mean = ts_log.mean(axis=1)
log_std = ts_log.std(axis=1)
ts_log_zscore = ts_log.sub(log_mean, axis=0).div(log_std.replace(0, 1), axis=0)

print('Preprocessing complete.')
print(f'  Raw:         shape={ts.shape}')
print(f'  Z-score:     shape={ts_zscore.shape}')
print(f'  Min-Max:     shape={ts_minmax.shape}')
print(f'  Log:         shape={ts_log.shape}')
print(f'  Log+Z-score: shape={ts_log_zscore.shape}')

### Comparison of Preprocessing Methods

Visualize the same household (near the median) under each preprocessing approach.

In [ ]:
# Pick a representative household near the median
median_idx = (household_mean - household_mean.median()).abs().idxmin()

fig, axes = plt.subplots(5, 1, figsize=(14, 16), sharex=True)
titles = ['Raw', 'Z-score', 'Min-Max', 'Log', 'Log + Z-score']
datasets = [ts, ts_zscore, ts_minmax, ts_log, ts_log_zscore]

for ax, title, data in zip(axes, titles, datasets):
    ax.plot(dates, data.loc[median_idx], linewidth=0.7)
    ax.set_title(f'{title} (median household, ID={ids.loc[median_idx]})')
    ax.set_ylabel('Value')

axes[-1].set_xlabel('Date')
plt.suptitle('Preprocessing Comparison: Median Household', fontsize=13)
plt.tight_layout()
plt.show()

**Observations:**
- **Z-score** and **Min-Max** remove absolute scale, making households comparable by pattern shape.
- **Log** compresses the high-consumption tail and makes the distribution more symmetric.
- **Log + Z-score** combines both benefits.

For clustering, we will primarily use **extracted features** (next section), which naturally handle scale differences. We also test clustering on z-score and log-transformed time series for comparison.

---

## 3. Feature Extraction

We extract 16 domain-relevant features from each household's daily time series. These features capture consumption level, variability, distribution shape, temporal dynamics, and periodicity.

| Feature | Description |
|---|---|
| `mean` | Average daily consumption (kWh) |
| `std` | Standard deviation of daily consumption |
| `cv` | Coefficient of variation (std / mean) -- relative variability |
| `min`, `max` | Minimum and maximum daily consumption |
| `skewness` | Asymmetry of the daily consumption distribution |
| `kurtosis` | Tailedness of the distribution (excess kurtosis) |
| `trend_slope` | Linear regression slope over the year (positive = increasing trend) |
| `winter_summer_ratio` | Mean winter (Dec-Feb) / mean summer (Jun-Aug) consumption |
| `peak_month` | Month (1-12) with highest average consumption |
| `weekend_weekday_ratio` | Weekend / weekday average consumption |
| `autocorr_lag1` | Autocorrelation at lag 1 day (day-to-day persistence) |
| `autocorr_lag7` | Autocorrelation at lag 7 days (weekly pattern) |
| `entropy` | Information entropy of the consumption histogram (regularity) |
| `zero_days` | Number of days with zero consumption |

In [ ]:
features = pd.DataFrame()
features['id'] = ids.values

# --- Basic statistics ---
features['mean'] = household_mean.values
features['std'] = household_std.values
features['cv'] = (household_std / household_mean.replace(0, np.nan)).fillna(0).values
features['min'] = household_min.values
features['max'] = household_max.values
features['skewness'] = ts.skew(axis=1).values
features['kurtosis'] = ts.kurtosis(axis=1).values

# --- Trend: linear regression slope over the year ---
day_numbers = np.arange(ts.shape[1], dtype=np.float64)
day_centered = day_numbers - day_numbers.mean()
denominator = np.sum(day_centered ** 2)
ts_centered = ts.values - ts.values.mean(axis=1, keepdims=True)
features['trend_slope'] = (ts_centered @ day_centered) / denominator

# --- Seasonality: winter/summer ratio ---
winter_cols = [c for c, d in zip(ts.columns, dates) if d.month in [12, 1, 2]]
summer_cols = [c for c, d in zip(ts.columns, dates) if d.month in [6, 7, 8]]
winter_mean = ts[winter_cols].mean(axis=1)
summer_mean = ts[summer_cols].mean(axis=1)
features['winter_summer_ratio'] = (winter_mean / summer_mean.replace(0, np.nan)).fillna(1).values

# --- Peak month ---
features['peak_month'] = monthly_data.idxmax(axis=1).map(
    {m: i + 1 for i, m in enumerate(month_names)}
).values

# --- Weekend/weekday ratio ---
day_of_week = dates.dayofweek  # Monday=0, Sunday=6
weekend_cols = [c for c, dow in zip(ts.columns, day_of_week) if dow >= 5]
weekday_cols = [c for c, dow in zip(ts.columns, day_of_week) if dow < 5]
weekend_mean = ts[weekend_cols].mean(axis=1)
weekday_mean = ts[weekday_cols].mean(axis=1)
features['weekend_weekday_ratio'] = (
    weekend_mean / weekday_mean.replace(0, np.nan)
).fillna(1).values

# --- Autocorrelation at lag 1 and lag 7 ---
def compute_autocorr(data, lag):
    """Compute autocorrelation at a given lag for each row."""
    n = data.shape[1]
    mean = data.mean(axis=1, keepdims=True)
    centered = data - mean
    var = np.sum(centered ** 2, axis=1)
    cov = np.sum(centered[:, :n - lag] * centered[:, lag:], axis=1)
    return np.where(var > 0, cov / var, 0)

ts_arr = ts.values
features['autocorr_lag1'] = compute_autocorr(ts_arr, 1)
features['autocorr_lag7'] = compute_autocorr(ts_arr, 7)

# --- Entropy (approximate, based on histogram of daily values) ---
def row_entropy(row, bins=20):
    counts, _ = np.histogram(row, bins=bins)
    probs = counts / counts.sum()
    probs = probs[probs > 0]
    return -np.sum(probs * np.log2(probs))

features['entropy'] = np.apply_along_axis(row_entropy, 1, ts_arr)

# --- Zero-day count ---
features['zero_days'] = (ts == 0).sum(axis=1).values

print(f'Extracted {features.shape[1] - 1} features for {features.shape[0]} households')
features.drop(columns=['id']).describe().round(3)

### Feature Distributions

In [ ]:
feat_cols = [c for c in features.columns if c != 'id']
n_feats = len(feat_cols)
ncols = 4
nrows = (n_feats + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3 * nrows))
axes_flat = axes.flatten()

for i, col in enumerate(feat_cols):
    axes_flat[i].hist(features[col], bins=60, edgecolor='black', alpha=0.7,
                      linewidth=0.3)
    axes_flat[i].set_title(col, fontsize=10)
    axes_flat[i].tick_params(labelsize=8)

for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle('Feature Distributions', fontsize=13)
plt.tight_layout()
plt.show()

### Feature Correlation Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
corr = features[feat_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax, center=0,
            xticklabels=True, yticklabels=True)
ax.set_title('Feature Correlation Matrix')
ax.tick_params(labelsize=8)
plt.tight_layout()
plt.show()

### Standardize Features for Clustering

Before clustering, all features are standardized to zero mean and unit variance. Extreme values in ratio features are capped at the 99th percentile to prevent distortion.

In [ ]:
feature_cols = [c for c in features.columns if c != 'id']
X_features_raw = features[feature_cols].values.copy()

# Handle inf/nan
X_features_raw = np.nan_to_num(X_features_raw, nan=0.0, posinf=0.0, neginf=0.0)

# Cap extreme values in ratio features
for col_idx, col_name in enumerate(feature_cols):
    if col_name in ['winter_summer_ratio', 'weekend_weekday_ratio']:
        p99 = np.percentile(X_features_raw[:, col_idx], 99)
        X_features_raw[:, col_idx] = np.clip(X_features_raw[:, col_idx], 0, p99)

# Standardize
scaler = StandardScaler()
X_features = scaler.fit_transform(X_features_raw)

print(f'Feature matrix for clustering: {X_features.shape}')

### Dimensionality Reduction (PCA)

PCA reveals how many independent dimensions the feature space effectively has.

In [ ]:
# PCA on extracted features
pca = PCA(n_components=min(10, X_features.shape[1]))
X_pca = pca.fit_transform(X_features)
explained_var = pca.explained_variance_ratio_
cumvar = np.cumsum(explained_var)
n_components_90 = int(np.argmax(cumvar >= 0.90)) + 1

# PCA on z-score time series
pca_ts = PCA(n_components=10)
X_ts_pca = pca_ts.fit_transform(ts_zscore.values)
ts_cumvar = np.cumsum(pca_ts.explained_variance_ratio_)

print(f'Features PCA: {n_components_90} components for 90% variance')
print(f'  First 5 components: {explained_var[:5].round(3)}')
print(f'Z-score TS PCA: 10 components explain {ts_cumvar[-1]*100:.1f}% variance')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(explained_var) + 1), explained_var, alpha=0.7,
            label='Individual')
axes[0].plot(range(1, len(explained_var) + 1), cumvar, 'ro-',
             label='Cumulative')
axes[0].axhline(y=0.90, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('PCA on Extracted Features')
axes[0].legend()

axes[1].bar(range(1, 11), pca_ts.explained_variance_ratio_, alpha=0.7,
            label='Individual')
axes[1].plot(range(1, 11), ts_cumvar, 'ro-', label='Cumulative')
axes[1].set_xlabel('Component')
axes[1].set_ylabel('Explained Variance Ratio')
axes[1].set_title('PCA on Z-score Time Series')
axes[1].legend()

plt.tight_layout()
plt.show()

---

## 4. Clustering

We evaluate three clustering algorithms:
1. **K-Means++** on extracted features (primary)
2. **K-Means++** on z-score time series (PCA-reduced) and log-transformed time series (PCA-reduced)
3. **Hierarchical clustering** (Ward linkage) on features
4. **DBSCAN** on PCA-reduced features

### 4.1 K-Means++ on Extracted Features (k=2..15)

In [ ]:
inertias_feat = []
silhouettes_feat = []
ch_scores_feat = []
db_scores_feat = []

print('K-Means++ on extracted features (standardized):')
print(f'{"k":>4} {"Inertia":>12} {"Silhouette":>11} {"CH":>8} {"DB":>8}')
print('-' * 50)

for k in K_RANGE:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, max_iter=300,
                random_state=RANDOM_STATE)
    labels = km.fit_predict(X_features)
    inertias_feat.append(km.inertia_)

    sil = silhouette_score(X_features, labels,
                           sample_size=min(5000, len(labels)),
                           random_state=RANDOM_STATE)
    silhouettes_feat.append(sil)
    ch_scores_feat.append(calinski_harabasz_score(X_features, labels))
    db_scores_feat.append(davies_bouldin_score(X_features, labels))

    print(f'{k:4d} {km.inertia_:12.0f} {sil:11.4f} '
          f'{ch_scores_feat[-1]:8.0f} {db_scores_feat[-1]:8.3f}')

### 4.2 K-Means++ on Time Series (PCA-reduced)

In [ ]:
# K-Means on z-score time series (PCA-10)
inertias_ts = []
silhouettes_ts = []
ch_scores_ts = []
db_scores_ts = []

print('K-Means++ on z-score time series (PCA-10):')
print(f'{"k":>4} {"Inertia":>12} {"Silhouette":>11} {"CH":>8} {"DB":>8}')
print('-' * 50)

for k in K_RANGE:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, max_iter=300,
                random_state=RANDOM_STATE)
    labels = km.fit_predict(X_ts_pca)
    inertias_ts.append(km.inertia_)

    sil = silhouette_score(X_ts_pca, labels,
                           sample_size=min(5000, len(labels)),
                           random_state=RANDOM_STATE)
    silhouettes_ts.append(sil)
    ch_scores_ts.append(calinski_harabasz_score(X_ts_pca, labels))
    db_scores_ts.append(davies_bouldin_score(X_ts_pca, labels))

    print(f'{k:4d} {km.inertia_:12.0f} {sil:11.4f} '
          f'{ch_scores_ts[-1]:8.0f} {db_scores_ts[-1]:8.3f}')

In [ ]:
# K-Means on log-transformed time series (PCA-10)
log_scaler = StandardScaler()
X_log_scaled = log_scaler.fit_transform(ts_log.values)
pca_log = PCA(n_components=10)
X_log_pca = pca_log.fit_transform(X_log_scaled)

inertias_log = []
silhouettes_log = []

print('K-Means++ on log-transformed time series (PCA-10):')
print(f'{"k":>4} {"Inertia":>12} {"Silhouette":>11}')
print('-' * 30)

for k in K_RANGE:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, max_iter=300,
                random_state=RANDOM_STATE)
    labels = km.fit_predict(X_log_pca)
    inertias_log.append(km.inertia_)

    sil = silhouette_score(X_log_pca, labels,
                           sample_size=min(5000, len(labels)),
                           random_state=RANDOM_STATE)
    silhouettes_log.append(sil)

    print(f'{k:4d} {km.inertia_:12.0f} {sil:11.4f}')

### Elbow and Silhouette Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Elbow: Features
axes[0, 0].plot(list(K_RANGE), inertias_feat, 'bo-', label='Features')
axes[0, 0].set_xlabel('k')
axes[0, 0].set_ylabel('Inertia')
axes[0, 0].set_title('Elbow Method: Features')
axes[0, 0].grid(True, alpha=0.3)

# Elbow: Time Series
axes[0, 1].plot(list(K_RANGE), inertias_ts, 'rs-', label='Z-score TS')
axes[0, 1].plot(list(K_RANGE), inertias_log, 'g^-', label='Log TS')
axes[0, 1].set_xlabel('k')
axes[0, 1].set_ylabel('Inertia')
axes[0, 1].set_title('Elbow Method: Time Series (PCA-10)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Silhouette comparison
axes[1, 0].plot(list(K_RANGE), silhouettes_feat, 'bo-', label='Features')
axes[1, 0].plot(list(K_RANGE), silhouettes_ts, 'rs-', label='Z-score TS')
axes[1, 0].plot(list(K_RANGE), silhouettes_log, 'g^-', label='Log TS')
axes[1, 0].set_xlabel('k')
axes[1, 0].set_ylabel('Silhouette Score')
axes[1, 0].set_title('Silhouette Score Comparison')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Calinski-Harabasz
axes[1, 1].plot(list(K_RANGE), ch_scores_feat, 'bo-', label='Features')
axes[1, 1].plot(list(K_RANGE), ch_scores_ts, 'rs-', label='Z-score TS')
axes[1, 1].set_xlabel('k')
axes[1, 1].set_ylabel('Calinski-Harabasz Score')
axes[1, 1].set_title('Calinski-Harabasz Index')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Clustering Evaluation: K-Means++ (k=2..15)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Determine optimal k
best_k_feat = list(K_RANGE)[np.argmax(silhouettes_feat)]
best_k_ts = list(K_RANGE)[np.argmax(silhouettes_ts)]

print(f'Best k for features (by silhouette): {best_k_feat} '
      f'(silhouette = {max(silhouettes_feat):.4f})')
print(f'Best k for z-score TS (by silhouette): {best_k_ts} '
      f'(silhouette = {max(silhouettes_ts):.4f})')

### 4.3 Final K-Means Clustering (optimal k on features)

In [ ]:
# Final K-Means with increased n_init for stability
km_final = KMeans(n_clusters=best_k_feat, init='k-means++', n_init=20,
                  max_iter=500, random_state=RANDOM_STATE)
labels_feat = km_final.fit_predict(X_features)
features['cluster_kmeans'] = labels_feat

# Cluster sizes
cluster_sizes = pd.Series(labels_feat).value_counts().sort_index()
print(f'K-Means++ with k={best_k_feat} on extracted features\n')
print('Cluster sizes:')
for c, n in cluster_sizes.items():
    print(f'  Cluster {c}: {n:,} households ({n / len(labels_feat) * 100:.1f}%)')

In [ ]:
# Cluster profiles (mean feature values)
cluster_profiles = features.groupby('cluster_kmeans')[feature_cols].mean()
cluster_profiles.round(3)

### 4.4 Hierarchical Clustering (Ward Linkage)

In [ ]:
hc = AgglomerativeClustering(n_clusters=best_k_feat, linkage='ward')
labels_hc = hc.fit_predict(X_features)
features['cluster_hierarchical'] = labels_hc

sil_hc = silhouette_score(X_features, labels_hc,
                          sample_size=min(5000, len(labels_hc)),
                          random_state=RANDOM_STATE)
ch_hc = calinski_harabasz_score(X_features, labels_hc)
db_hc = davies_bouldin_score(X_features, labels_hc)

print(f'Hierarchical Clustering (Ward, k={best_k_feat}):')
print(f'  Silhouette:        {sil_hc:.4f}')
print(f'  Calinski-Harabasz: {ch_hc:.0f}')
print(f'  Davies-Bouldin:    {db_hc:.3f}')

hc_sizes = pd.Series(labels_hc).value_counts().sort_index()
print('\nCluster sizes:')
for c, n in hc_sizes.items():
    print(f'  Cluster {c}: {n:,} households ({n / len(labels_hc) * 100:.1f}%)')

### 4.5 DBSCAN

DBSCAN is a density-based algorithm that does not require specifying k. We first use the k-distance graph to select an appropriate `eps` value, then try multiple values.

In [ ]:
# Use PCA-reduced features for DBSCAN (works better in lower dimensions)
X_feat_pca = PCA(n_components=n_components_90).fit_transform(X_features)

# k-distance graph for eps selection
nn = NearestNeighbors(n_neighbors=10)
nn.fit(X_feat_pca)
distances, _ = nn.kneighbors(X_feat_pca)
k_dist = np.sort(distances[:, -1])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_dist)
ax.set_xlabel('Points (sorted)')
ax.set_ylabel('10-NN Distance')
ax.set_title('k-Distance Graph for DBSCAN eps Selection')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Try DBSCAN with several eps values
best_dbscan_sil = -1
best_dbscan_eps = None
best_dbscan_labels = None

print(f'{"eps":>5} {"Clusters":>9} {"Noise":>8} {"Noise%":>8} {"Silhouette":>11}')
print('-' * 48)

for eps in [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]:
    db = DBSCAN(eps=eps, min_samples=10)
    labels_db = db.fit_predict(X_feat_pca)
    n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
    n_noise = (labels_db == -1).sum()

    if n_clusters >= 2:
        mask = labels_db != -1
        sil_db = silhouette_score(X_feat_pca[mask], labels_db[mask],
                                  sample_size=min(5000, mask.sum()),
                                  random_state=RANDOM_STATE)
    else:
        sil_db = -1

    print(f'{eps:5.1f} {n_clusters:9d} {n_noise:8d} '
          f'{n_noise / len(labels_db) * 100:7.1f}% {sil_db:11.4f}')

    if sil_db > best_dbscan_sil and n_clusters >= 2:
        best_dbscan_sil = sil_db
        best_dbscan_eps = eps
        best_dbscan_labels = labels_db.copy()

if best_dbscan_labels is not None:
    features['cluster_dbscan'] = best_dbscan_labels
    print(f'\nBest DBSCAN: eps={best_dbscan_eps}, silhouette={best_dbscan_sil:.4f}')
    dbscan_sizes = pd.Series(best_dbscan_labels).value_counts().sort_index()
    for c, n in dbscan_sizes.items():
        label = 'NOISE' if c == -1 else f'Cluster {c}'
        print(f'  {label}: {n:,} households ({n / len(best_dbscan_labels) * 100:.1f}%)')
else:
    print('\nDBSCAN did not find a valid clustering.')

**DBSCAN observation:** DBSCAN tends to produce degenerate results on this dataset -- it places the vast majority of data into one cluster and identifies only a handful of outlier points as a second cluster. This is because the feature space is continuous and unimodal (most households are "typical"), making density-based methods unable to find meaningful natural boundaries.

### 4.6 K-Means on Z-score Time Series (for comparison)

In [ ]:
km_ts = KMeans(n_clusters=best_k_ts, init='k-means++', n_init=20,
               max_iter=500, random_state=RANDOM_STATE)
labels_ts = km_ts.fit_predict(X_ts_pca)
features['cluster_ts_kmeans'] = labels_ts

sil_ts = silhouette_score(X_ts_pca, labels_ts,
                          sample_size=min(5000, len(labels_ts)),
                          random_state=RANDOM_STATE)
ch_ts = calinski_harabasz_score(X_ts_pca, labels_ts)
db_ts = davies_bouldin_score(X_ts_pca, labels_ts)

ts_sizes = pd.Series(labels_ts).value_counts().sort_index()
print(f'K-Means on z-score time series (k={best_k_ts}):')
print(f'  Silhouette:        {sil_ts:.4f}')
print(f'  Calinski-Harabasz: {ch_ts:.0f}')
print(f'  Davies-Bouldin:    {db_ts:.3f}')
print('\nCluster sizes:')
for c, n in ts_sizes.items():
    print(f'  Cluster {c}: {n:,} households ({n / len(labels_ts) * 100:.1f}%)')

---

## 5. Results & Analysis

### 5.1 Algorithm Comparison

In [ ]:
# Compute final metrics for K-Means on features
sil_km_feat = silhouette_score(X_features, labels_feat,
                                sample_size=min(5000, len(labels_feat)),
                                random_state=RANDOM_STATE)
ch_km_feat = calinski_harabasz_score(X_features, labels_feat)
db_km_feat = davies_bouldin_score(X_features, labels_feat)

comparison_data = {
    'Method': ['K-Means++ (Features)', 'Hierarchical/Ward (Features)',
               f'K-Means++ (Z-score TS, k={best_k_ts})'],
    'k': [best_k_feat, best_k_feat, best_k_ts],
    'Silhouette': [round(sil_km_feat, 4), round(sil_hc, 4), round(sil_ts, 4)],
    'Calinski-Harabasz': [round(ch_km_feat, 0), round(ch_hc, 0), round(ch_ts, 0)],
    'Davies-Bouldin': [round(db_km_feat, 3), round(db_hc, 3), round(db_ts, 3)],
}

if best_dbscan_labels is not None:
    comparison_data['Method'].append(f'DBSCAN (eps={best_dbscan_eps})')
    comparison_data['k'].append(
        len(set(best_dbscan_labels)) - (1 if -1 in best_dbscan_labels else 0))
    comparison_data['Silhouette'].append(round(best_dbscan_sil, 4))
    comparison_data['Calinski-Harabasz'].append('--')
    comparison_data['Davies-Bouldin'].append('--')

comparison_df = pd.DataFrame(comparison_data)
comparison_df

In [ ]:
# Bar chart comparison
methods = ['K-Means\n(Features)', 'Hierarchical\n(Features)',
           'K-Means\n(Z-score TS)']
sils = [sil_km_feat, sil_hc, sil_ts]
dbs = [db_km_feat, db_hc, db_ts]

if best_dbscan_labels is not None:
    methods.append(f'DBSCAN\n(eps={best_dbscan_eps})')
    sils.append(best_dbscan_sil)
    mask_db = best_dbscan_labels != -1
    if mask_db.sum() > best_k_feat:
        dbs.append(davies_bouldin_score(
            X_feat_pca[mask_db], best_dbscan_labels[mask_db]))
    else:
        dbs.append(0)

colors = ['steelblue', 'coral', 'seagreen', 'mediumpurple']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(methods, sils, color=colors[:len(methods)], edgecolor='black')
axes[0].set_ylabel('Silhouette Score (higher is better)')
axes[0].set_title('Silhouette Score Comparison')
for i, v in enumerate(sils):
    axes[0].text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=10)

axes[1].bar(methods, dbs, color=colors[:len(methods)], edgecolor='black')
axes[1].set_ylabel('Davies-Bouldin Index (lower is better)')
axes[1].set_title('Davies-Bouldin Index Comparison')
for i, v in enumerate(dbs):
    axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=10)

plt.suptitle('Clustering Algorithm Comparison', fontsize=13)
plt.tight_layout()
plt.show()

### 5.2 Cluster Consumption Profiles

Mean consumption profile over the year for each K-Means cluster, with IQR bands.

In [ ]:
fig, axes = plt.subplots(best_k_feat, 1, figsize=(14, 4 * best_k_feat),
                         sharex=True)
if best_k_feat == 1:
    axes = [axes]

for c in range(best_k_feat):
    mask = labels_feat == c
    cluster_ts = ts.values[mask]
    mean_profile = cluster_ts.mean(axis=0)
    q25 = np.percentile(cluster_ts, 25, axis=0)
    q75 = np.percentile(cluster_ts, 75, axis=0)

    axes[c].plot(dates, mean_profile, linewidth=1.5, label='Mean')
    axes[c].fill_between(dates, q25, q75, alpha=0.2, label='IQR')
    axes[c].set_ylabel('kWh')
    axes[c].set_title(f'Cluster {c} (n={mask.sum():,}, '
                      f'avg={features.loc[mask, "mean"].mean():.1f} kWh/day)')
    axes[c].legend(loc='upper right')
    axes[c].grid(True, alpha=0.2)

axes[-1].set_xlabel('Date')
plt.suptitle(f'K-Means Cluster Profiles (k={best_k_feat}, features)',
             fontsize=13)
plt.tight_layout()
plt.show()

### 5.3 t-SNE Visualization

2D t-SNE projection of the feature space, colored by cluster assignment. A random subsample of 5,000 households is used for computational efficiency.

In [ ]:
np.random.seed(RANDOM_STATE)
n_sample = min(5000, len(X_features))
sample_idx = np.random.choice(len(X_features), n_sample, replace=False)

print(f'Running t-SNE on {n_sample} samples (this may take a minute)...')
tsne = TSNE(n_components=2, perplexity=30, random_state=RANDOM_STATE,
            max_iter=1000)
X_tsne = tsne.fit_transform(X_features[sample_idx])
print('Done.')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# K-Means labels
scatter1 = axes[0].scatter(X_tsne[:, 0], X_tsne[:, 1],
                           c=labels_feat[sample_idx], cmap='tab10',
                           alpha=0.5, s=5)
axes[0].set_title(f't-SNE: K-Means (k={best_k_feat}, features)')
axes[0].set_xlabel('t-SNE 1')
axes[0].set_ylabel('t-SNE 2')
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

# Hierarchical labels
scatter2 = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1],
                           c=labels_hc[sample_idx], cmap='tab10',
                           alpha=0.5, s=5)
axes[1].set_title(f't-SNE: Hierarchical (k={best_k_feat}, Ward)')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
plt.colorbar(scatter2, ax=axes[1], label='Cluster')

plt.suptitle(f't-SNE Visualization of Clusters ({n_sample} sample)',
             fontsize=13)
plt.tight_layout()
plt.show()

### 5.4 PCA 2D Visualization

In [ ]:
pca2d = PCA(n_components=2)
X_pca2d = pca2d.fit_transform(X_features)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

scatter1 = axes[0].scatter(X_pca2d[:, 0], X_pca2d[:, 1],
                           c=labels_feat, cmap='tab10', alpha=0.3, s=3)
axes[0].set_title(f'PCA: K-Means (k={best_k_feat})')
axes[0].set_xlabel(f'PC1 ({pca2d.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca2d.explained_variance_ratio_[1]*100:.1f}%)')
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

scatter2 = axes[1].scatter(X_pca2d[:, 0], X_pca2d[:, 1],
                           c=labels_hc, cmap='tab10', alpha=0.3, s=3)
axes[1].set_title(f'PCA: Hierarchical (k={best_k_feat})')
axes[1].set_xlabel(f'PC1 ({pca2d.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca2d.explained_variance_ratio_[1]*100:.1f}%)')
plt.colorbar(scatter2, ax=axes[1], label='Cluster')

plt.suptitle('PCA 2D Visualization of Clusters', fontsize=13)
plt.tight_layout()
plt.show()

### 5.5 Feature Distributions per Cluster (Boxplots)

In [ ]:
key_features = ['mean', 'std', 'cv', 'winter_summer_ratio', 'trend_slope',
                'weekend_weekday_ratio', 'autocorr_lag1', 'peak_month']

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes_flat = axes.flatten()

for i, feat in enumerate(key_features):
    data_by_cluster = [features.loc[labels_feat == c, feat].values
                       for c in range(best_k_feat)]
    bp = axes_flat[i].boxplot(data_by_cluster,
                              labels=[str(c) for c in range(best_k_feat)],
                              patch_artist=True)
    colors_bp = plt.cm.tab10(np.linspace(0, 1, best_k_feat))
    for patch, color in zip(bp['boxes'], colors_bp):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    axes_flat[i].set_title(feat)
    axes_flat[i].set_xlabel('Cluster')
    axes_flat[i].grid(True, alpha=0.2)

plt.suptitle(f'Feature Distributions per Cluster (K-Means, k={best_k_feat})',
             fontsize=13)
plt.tight_layout()
plt.show()

### 5.6 Monthly Consumption Heatmap by Cluster

In [ ]:
# Compute monthly average consumption per cluster
month_labels = []
monthly_profiles = []

for m in range(1, 13):
    col_mask = [i for i, d in enumerate(dates) if d.month == m]
    month_labels.append(dates[col_mask[0]].strftime('%b'))
    monthly_profiles.append(
        [ts.values[labels_feat == c][:, col_mask].mean()
         for c in range(best_k_feat)]
    )

monthly_heatmap = np.array(monthly_profiles).T  # shape: (k, 12)

fig, ax = plt.subplots(figsize=(12, max(4, best_k_feat)))
sns.heatmap(monthly_heatmap, annot=True, fmt='.1f', cmap='YlOrRd',
            xticklabels=month_labels,
            yticklabels=[f'Cluster {c} (n={cluster_sizes[c]:,})'
                         for c in range(best_k_feat)],
            ax=ax)
ax.set_title('Average Monthly Consumption by Cluster (kWh)')
ax.set_xlabel('Month')
plt.tight_layout()
plt.show()

---

## 6. Summary & Findings

### Optimal Clustering

**K-Means++ on extracted features with k=3** is the best approach, providing the highest silhouette score and the most interpretable cluster structure.

### Cluster Descriptions

| Cluster | Name | Size | Avg kWh/day | Key Characteristics |
|---|---|---|---|---|
| 0 | Low/Irregular | ~3% | ~1.2 | Very low consumption, high variability (CV~2.5), many zero-days (~218), intermittent occupancy |
| 1 | High Consumers | ~17% | ~21.7 | High stable consumption, strong autocorrelation (lag1~0.81), slight downward trend, likely electric heating |
| 2 | Typical/Moderate | ~80% | ~6.7 | Moderate consumption, mild seasonality, moderate variability, standard residential patterns |

### Key Takeaways

1. **Three natural consumption segments exist** in the data, driven primarily by consumption *level* and secondarily by consumption *regularity*.

2. **Feature-based clustering outperforms direct time series clustering.** Extracting domain-relevant features (mean, variability, seasonality, autocorrelation) and clustering on those produces more interpretable and better-separated clusters than clustering on raw or normalized time series.

3. **DBSCAN is not suitable** for this data. The continuous, unimodal nature of the feature space means density-based methods degenerate into one giant cluster.

4. **Hierarchical clustering produces similar but slightly worse results** compared to K-Means++ (silhouette ~0.31 vs ~0.40), with less balanced cluster sizes.

5. **Implications for forecasting:** Cluster-specific models can be trained:
   - **Cluster 0 (Low/Irregular):** Simple model, or flag as unpredictable
   - **Cluster 1 (High):** High-capacity model exploiting strong temporal regularity (autocorrelation ~0.8)
   - **Cluster 2 (Typical):** Standard model for the majority of households

In [ ]:
# Final summary statistics
print('=' * 60)
print('CLUSTERING SUMMARY')
print('=' * 60)
print(f'Best approach: K-Means++ on extracted features')
print(f'Optimal k: {best_k_feat}')
print(f'Silhouette score: {sil_km_feat:.4f}')
print(f'Calinski-Harabasz: {ch_km_feat:.0f}')
print(f'Davies-Bouldin: {db_km_feat:.4f}')
print(f'\nCluster sizes:')
for c, n in cluster_sizes.items():
    print(f'  Cluster {c}: {n:,} ({n/len(labels_feat)*100:.1f}%)')